# Laboratorio 1 — Flujo de un modelo supervisado (15 min)
**Curso:** Aprendizaje Automático · MCDA 2026-1  
**Sesión:** 01 — Introducción, métricas y validación cruzada

## Objetivo
Entrenar tu primer clasificador, medir su desempeño con varias métricas y validar la estabilidad del resultado con K-Fold CV.

## Dataset
Usaremos **Breast Cancer Wisconsin** (incluido en `sklearn`): 569 pacientes, 30 variables numéricas, variable respuesta binaria (0 = maligno, 1 = benigno).

## Agenda (15 min)
| Paso | Tiempo | Qué haces |
|------|--------|-----------|
| 1. Cargar y explorar | 2 min | Entender forma de los datos |
| 2. Partición train/test | 1 min | `train_test_split` estratificado |
| 3. Entrenar baseline | 2 min | Regresión logística |
| 4. Métricas | 3 min | Accuracy, precision, recall, F1, matriz de confusión, ROC-AUC |
| 5. Validación cruzada | 2 min | K-Fold con k=5 |
| 6. Retos | 5 min | 3 micro-retos individuales |


## Paso 1 · Cargar y explorar los datos (2 min)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target  # 0 = maligno, 1 = benigno

print('Forma X:', X.shape)
print('Clases:', dict(zip(*np.unique(y, return_counts=True))))
X.head()

**Pregunta rápida:** ¿El dataset está balanceado? ¿Qué métrica te preocuparía si estuviera muy desbalanceado?

## Paso 2 · Partición train / test (1 min)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,      # mantiene proporción de clases en train y test
    random_state=42  # reproducibilidad
)
print(f'Train: {X_train.shape[0]} filas · Test: {X_test.shape[0]} filas')

## Paso 3 · Entrenar el modelo baseline (2 min)
Regresión logística como modelo base. Estandarizamos porque es un modelo lineal sensible a la escala.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=5000, random_state=42))
])

pipe.fit(X_train, y_train)
print('Modelo entrenado ✔')

## Paso 4 · Métricas de desempeño (3 min)
Evaluamos sobre el **conjunto de test** (nunca visto durante el entrenamiento).

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay)

y_pred  = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]

print(f'Accuracy : {accuracy_score(y_test, y_pred):.3f}')
print(f'Precision: {precision_score(y_test, y_pred):.3f}')
print(f'Recall   : {recall_score(y_test, y_pred):.3f}')
print(f'F1       : {f1_score(y_test, y_pred):.3f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_proba):.3f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=['Maligno', 'Benigno']
).plot(ax=ax[0], colorbar=False)
ax[0].set_title('Matriz de confusión')

RocCurveDisplay.from_predictions(y_test, y_proba, ax=ax[1])
ax[1].plot([0, 1], [0, 1], '--', color='gray', label='Aleatorio')
ax[1].set_title('Curva ROC')
ax[1].legend()
plt.tight_layout(); plt.show()

## Paso 5 · Validación cruzada K-Fold (2 min)
Una sola partición puede ser optimista o pesimista por azar. Con K-Fold estimamos el error promedio y su desviación.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv, scoring='f1')

print('F1 por fold:', np.round(scores, 3))
print(f'F1 promedio: {scores.mean():.3f} ± {scores.std():.3f}')

## Paso 6 · Retos individuales (5 min)

> Completa los 3 retos en las celdas de abajo. Son rápidos: se trata de modificar una línea y comparar.

### Reto 1 · Romper la estratificación
Vuelve a partir los datos **sin** `stratify=y` y con `random_state=7`. ¿Cambia mucho la proporción de clases en test? Compara el F1 resultante con el de arriba.

### Reto 2 · Cambiar la métrica objetivo
Este dataset marca maligno como clase 0. Si el costo de **no detectar un tumor maligno** es alto, ¿qué métrica reportarías al hospital? Calcúlala tratando la clase maligna (0) como la positiva (`pos_label=0` en `recall_score` / `precision_score`).

### Reto 3 · Cambiar k en K-Fold
Corre la validación cruzada con `n_splits=10`. ¿Subió o bajó la desviación estándar del F1? Explica en una línea por qué.

In [ ]:
# Reto 1 — tu código aquí


In [ ]:
# Reto 2 — tu código aquí
# pista: recall_score(y_test, y_pred, pos_label=0)


In [ ]:
# Reto 3 — tu código aquí


## Cierre
Con este flujo ya tienes el esqueleto de cualquier proyecto supervisado: **partir los datos → entrenar → medir con varias métricas → validar con CV**.